# GTSRB — Baseline vs Virtual Adversarial Training
Dataset: German Traffic Sign Recognition Benchmark (Kaggle)  
Architectures: **FCN** (fully connected) and **SmallCNN** (VGG-style)  
Modes: **baseline** (supervised cross-entropy) and **vat** (+ VAT regularisation)  

Change `Config` fields in the experiment cells — all training logic is shared.

In [ ]:
import os, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
from PIL import Image
from dataclasses import dataclass
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {DEVICE}')

In [ ]:
import kagglehub

DATASET_ROOT = kagglehub.dataset_download('meowmeowmeowmeowmeow/gtsrb-german-traffic-sign')
print(f'Dataset root: {DATASET_ROOT}')

## Config

In [ ]:
@dataclass
class Config:
    # Dataset paths — derived from the kagglehub download above
    train_dir: str    = os.path.join(DATASET_ROOT, 'Train')
    test_dir: str     = os.path.join(DATASET_ROOT, 'Test')
    test_csv: str     = os.path.join(DATASET_ROOT, 'Test.csv')

    # Image
    img_size: int     = 48
    num_classes: int  = 43

    # Model
    model_type: str   = 'FCN'       # 'FCN' | 'SmallCNN'
    dropout: float    = 0.5

    # Training
    mode: str         = 'baseline'  # 'baseline' | 'vat'
    batch_size: int   = 128
    epochs: int       = 20
    lr: float         = 1e-3
    weight_decay: float = 1e-4
    val_fraction: float = 0.15
    patience: int     = 15          # early-stop patience (epochs without val improvement)

    # VAT
    vat_eps: float    = 1.0         # L2 perturbation budget
    vat_xi: float     = 1e-6        # finite-difference step for power iteration
    vat_num_iters: int = 1          # power-iteration steps (1 is standard)
    vat_alpha: float  = 1.0         # weight of VAT term in total loss

    # Misc# Semi-supervised
    labeled_fraction: float = 1.0   # 1.0 = fully supervised, 0.0 = unsupervised
    seed: int         = 42
    num_workers: int  = 0

# Confirm paths are set correctly
print(Config())

## Model Architectures

In [ ]:
class FCN(nn.Module):
    """
    Two-hidden-layer fully connected network.
    Architecture: in -> 1200 -> 1200 -> num_classes  (BN + ReLU + Dropout).
    Matches the FCN used in the VAT paper (Miyato et al., TPAMI 2019).
    """
    def __init__(self, in_features: int, num_classes: int, dropout: float = 0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 1200), nn.BatchNorm1d(1200), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1200, 1200),        nn.BatchNorm1d(1200), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1200, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class SmallCNN(nn.Module):
    """
    Small VGG-style CNN for traffic sign images (3 x img_size x img_size).
    3 conv blocks (2 conv each) + FC head = 7 weight layers total.
    For img_size=48: spatial dims 48->24->12->6, so FC input = 128*6*6.
    """
    def __init__(self, num_classes: int = 43, dropout: float = 0.5):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 48x48 -> 24x24
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.25),
            # Block 2: 24x24 -> 12x12
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.25),
            # Block 3: 12x12 -> 6x6
            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


def build_model(cfg: Config) -> nn.Module:
    if cfg.model_type == 'FCN':
        in_features = 3 * cfg.img_size * cfg.img_size
        return FCN(in_features, cfg.num_classes, cfg.dropout)
    elif cfg.model_type == 'SmallCNN':
        return SmallCNN(cfg.num_classes, cfg.dropout)
    else:
        raise ValueError(f'Unknown model_type: {cfg.model_type}')


# Smoke test
for mt in ('FCN', 'SmallCNN'):
    cfg_tmp = Config(model_type=mt)
    m   = build_model(cfg_tmp)
    out = m(torch.randn(4, 3, 48, 48))
    p   = sum(x.numel() for x in m.parameters())
    print(f'  {mt:10}  out={tuple(out.shape)}  params={p:,}')

## Data Loading

In [ ]:
SENTINEL = -1   # label value used for unlabelled training samples


class SemiSupervisedSubset(Dataset):
    """
    Wraps a Subset (from random_split); masks a fraction of labels with SENTINEL.
    Val / test sets are never wrapped — they remain fully labelled.
    """
    def __init__(self, subset: Dataset, labeled_fraction: float, seed: int):
        self.subset = subset
        n = len(subset)
        n_labelled = int(n * labeled_fraction)
        rng = np.random.default_rng(seed)
        idx = np.arange(n)
        rng.shuffle(idx)
        self._labelled = set(idx[:n_labelled].tolist())
        print(f'  GTSRB  train={n:,} '
              f'(labelled={n_labelled:,}, unlabelled={n - n_labelled:,})')

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        x, y = self.subset[i]
        return x, (y if i in self._labelled else SENTINEL)


# Pre-computed per-channel stats for GTSRB
GTSRB_MEAN = (0.3337, 0.3064, 0.3171)
GTSRB_STD  = (0.2672, 0.2564, 0.2629)


class GTSRBTestDataset(Dataset):
    """Test set: flat image folder + CSV with 'Path' and 'ClassId' columns."""
    def __init__(self, csv_path: str, img_dir: str, transform=None):
        self.df        = pd.read_csv(csv_path)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(os.path.join(self.img_dir, os.path.basename(row['Path']))).convert('RGB')
        label = int(row['ClassId'])
        if self.transform:
            img = self.transform(img)
        return img, label


def get_loaders(cfg: Config):
    torch.manual_seed(cfg.seed)

    train_tf = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.RandomHorizontalFlip(p=0.1),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(GTSRB_MEAN, GTSRB_STD),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor(),
        transforms.Normalize(GTSRB_MEAN, GTSRB_STD),
    ])

    # Use ImageFolder for the class-per-subfolder train layout
    full_aug   = ImageFolder(cfg.train_dir, transform=train_tf)
    full_clean = ImageFolder(cfg.train_dir, transform=val_tf)
    test_ds    = GTSRBTestDataset(cfg.test_csv, cfg.test_dir, transform=val_tf)

    n_val   = int(len(full_aug) * cfg.val_fraction)
    n_train = len(full_aug) - n_val
    gen     = torch.Generator().manual_seed(cfg.seed)

    # Same seed -> identical index split; augmented for train, clean for val
    train_ds, _ = random_split(full_aug,   [n_train, n_val], generator=gen)
    _, val_ds   = random_split(full_clean, [n_train, n_val],
                               generator=torch.Generator().manual_seed(cfg.seed))

    use_pin = torch.cuda.is_available()   # only pin memory when CUDA is actually present
    kw = dict(
        num_workers=cfg.num_workers,
        pin_memory=use_pin,
        persistent_workers=(cfg.num_workers > 0)
    )
    semi_train_ds = SemiSupervisedSubset(train_ds, cfg.labeled_fraction, cfg.seed)
    print(f'  val={len(val_ds):,}  test={len(test_ds):,}')

    train_loader = DataLoader(semi_train_ds, batch_size=cfg.batch_size, shuffle=True,  drop_last=True, **kw)
    val_loader   = DataLoader(val_ds,        batch_size=cfg.batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,       batch_size=cfg.batch_size, shuffle=False, **kw)

    return train_loader, val_loader, test_loader

## VAT Loss

In [ ]:
def _l2_normalize(t: torch.Tensor) -> torch.Tensor:
    """Unit-L2-normalise each sample in the batch."""
    flat = t.view(t.size(0), -1)
    norm = flat.norm(p=2, dim=1, keepdim=True).clamp(min=1e-8)
    return (flat / norm).view_as(t)


def vat_loss(model: nn.Module, x: torch.Tensor,
             eps: float = 1.0, xi: float = 1e-6, num_iters: int = 1) -> torch.Tensor:
    """
    Virtual Adversarial Training regularisation loss.
    Miyato et al., 'Virtual Adversarial Training', TPAMI 2019.

    Returns  KL( p(y|x; theta) || p(y|x+r_adv; theta) )
    where r_adv is the worst-case L2-bounded perturbation found via
    power iteration (num_iters=1 is standard and sufficient in practice).
    """
    # Clean distribution — treated as a fixed target, no gradient
    with torch.no_grad():
        p_clean = F.softmax(model(x), dim=1)

    # Random unit-norm starting direction
    d = _l2_normalize(torch.randn_like(x))

    # Power iteration: find direction that maximises KL divergence
    for _ in range(num_iters):
        d = d.detach().requires_grad_(True)
        kl = F.kl_div(
            F.log_softmax(model(x + xi * d), dim=1),
            p_clean, reduction='batchmean'
        )
        # Gradient only w.r.t. d — model-param gradients are NOT accumulated here
        (grad_d,) = torch.autograd.grad(kl, d)
        d = _l2_normalize(grad_d.detach())

    # Final VAT loss — gradient flows to model params via this forward pass
    x_adv = (x + eps * d).detach()
    return F.kl_div(
        F.log_softmax(model(x_adv), dim=1),
        p_clean.detach(), reduction='batchmean'
    )

## Trainer
A single `train()` function handles both modes — switch `cfg.mode` to change behaviour.

In [ ]:
def _train_epoch(model, loader, criterion, optimizer, cfg: Config):
    """
    Executes one complete training pass over the provided dataset.
    Updates model parameters based on the calculated loss.
    Displays a text-based progress bar in the console.
    """
    model.train()
    total_loss = total_acc = 0.0
    n = len(loader)

    for i, (x, y) in enumerate(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)

        # Semi-supervised: cross-entropy only on labelled samples
        labelled = (y != SENTINEL)
        if labelled.any():
            loss = criterion(logits[labelled], y[labelled])
        else:
            loss = torch.zeros(1, device=DEVICE, requires_grad=True)

        if cfg.mode == 'vat':
            loss = loss + cfg.vat_alpha * vat_loss(
                model, x, eps=cfg.vat_eps, xi=cfg.vat_xi, num_iters=cfg.vat_num_iters
            )

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        if labelled.any():
            total_acc += (logits[labelled].argmax(1) == y[labelled]).float().mean().item()

        # Calculates the percentage of completion for the current epoch.
        progress = (i + 1) / n

        # Determines the number of filled blocks in a 30-character wide bar.
        filled_length = int(30 * progress)

        # Constructs the visual string for the progress bar.
        bar = '=' * filled_length + '-' * (30 - filled_length)

        # Outputs the bar to the console.
        # The \r character overwrites the previous line.
        print(f'\r  Step {i + 1:>3}/{n} [{bar}] Loss: {loss.item():.4f}', end='', flush=True)

    # Clears the progress bar line after the epoch completes to preserve table formatting.
    print('\r' + ' ' * 70 + '\r', end='', flush=True)

    return total_loss / n, total_acc / n


@torch.no_grad()
def _evaluate(model, loader, criterion):
    model.eval()
    total_loss = total_acc = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits      = model(x)
        total_loss += criterion(logits, y).item()
        total_acc  += (logits.argmax(1) == y).float().mean().item()
    n = len(loader)
    return total_loss / n, total_acc / n


def train(cfg: Config):
    """Train for one Config; return (history dict, final metrics dict)."""
    torch.manual_seed(cfg.seed)

    train_loader, val_loader, test_loader = get_loaders(cfg)
    model     = build_model(cfg).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.lr * 1e-2)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = -1.0
    best_state   = None
    no_improve   = 0

    label = f'[{cfg.mode.upper()}] {cfg.model_type} | GTSRB'
    print(f'\n{"="*60}\n  {label}\n{"="*60}')
    print(f'  {"Ep":>3} | {"TrLoss":>8} {"TrAcc":>7} | {"VlLoss":>8} {"VlAcc":>7} |')
    print(f'  {"-"*50}')

    for epoch in range(1, cfg.epochs + 1):
        tr_l, tr_a = _train_epoch(model, train_loader, criterion, optimizer, cfg)
        vl_l, vl_a = _evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_l)
        history['train_acc'].append(tr_a)
        history['val_loss'].append(vl_l)
        history['val_acc'].append(vl_a)

        tag = ''
        if vl_a > best_val_acc:
            best_val_acc = vl_a
            best_state   = copy.deepcopy(model.state_dict())
            no_improve   = 0
            tag = ' *'
        else:
            no_improve += 1

        if epoch % 10 == 0 or epoch == 1 or tag:
            print(f'  {epoch:>3} | {tr_l:>8.4f} {tr_a*100:>6.2f}% | {vl_l:>8.4f} {vl_a*100:>6.2f}% |{tag}')

        if no_improve >= cfg.patience:
            print(f'\n  Early stop at epoch {epoch} (no val gain for {cfg.patience} epochs)')
            break

    model.load_state_dict(best_state)
    ts_l, ts_a = _evaluate(model, test_loader, criterion)
    print(f'\n  Best Val Acc : {best_val_acc*100:.2f}%')
    print(f'  Test Acc     : {ts_a*100:.2f}%  |  Test Loss: {ts_l:.4f}')

    return history, {'val_acc': best_val_acc, 'test_acc': ts_a, 'test_loss': ts_l}

## Experiments
Four runs: FCN and SmallCNN, each in baseline and VAT modes.  
Adjust any `Config` field (e.g. `vat_eps`, `vat_alpha`, `epochs`) to explore hyperparameters.

In [ ]:
import pickle

# Defines the subset of training data fractions to be evaluated.
FRACTIONS = [0.0, 0.25, 0.5, 1.0]

# Initializes a dictionary to store performance metrics and histories for each data fraction.
sweep = {}

# Defines the combinations of neural network architectures and training modes to evaluate.
RUNS = [
    ('FCN',      'vat'),
    ('FCN',      'baseline'),
    ('SmallCNN', 'vat'),
    ('SmallCNN', 'baseline'),
]

# Iterates over each defined data fraction to conduct training experiments.
for frac in FRACTIONS:
    print(f'\n{"#"*60}')
    print(f'  labeled_fraction = {frac:.0%}')
    print(f'{"#"*60}')
    sweep.setdefault(frac, {})

    # Iterates over each model architecture and training mode combination.
    for model_type, mode in RUNS:
        if mode == 'baseline' and frac == 0.0:
            continue

        # Determines the text description for the current data fraction.
        if frac == 0.0:
            fraction_text = "no labelled data"
        elif frac == 1.0:
            fraction_text = "fully labelled data"
        else:
            fraction_text = f"{frac:.0%} partially labelled data"

        # Outputs the current experiment configuration to the console.
        print(f"  -> Model: {model_type} | Mode: {mode} | Fraction: {fraction_text}")

        key = f'{model_type} / {mode}'

        # Executes the training process and stores both history and final metrics.
        h, m = train(Config(model_type=model_type, mode=mode, labeled_fraction=frac))
        sweep[frac][key] = {
            'history': h,
            'metrics': m
        }

# Serializes the nested dictionary to a binary file using pickle for persistent storage.
with open('NoteBook_GTSRB_Data.pkl', 'wb') as file:
    pickle.dump(sweep, file)
print("\nSaved training results and history to NoteBook_GTSRB_Data.pkl")

## Results

In [ ]:
import matplotlib.pyplot as plt
import pickle
import os

# Deserializes the dictionary from the binary file if it is not already loaded in memory.
# The 'rb' flag opens the file in read-binary mode.
if 'sweep' not in locals():
    if os.path.exists('NoteBook_Artificial_Data.pkl'):
        with open('NoteBook_Artificial_Data.pkl', 'rb') as file:
            sweep = pickle.load(file)
        print("Data successfully loaded from sweep_data.pkl\n")
    else:
        raise FileNotFoundError("The file sweep_data.pkl does not exist. Run the training script first.")


# ── Summary table ─────────────────────────────────────────────────────
KEYS = ['FCN / vat', 'FCN / baseline', 'SmallCNN / vat', 'SmallCNN / baseline']

col_w = 14
header = f'  {"Labeled %":>18}' + ''.join(f'{k:>{col_w}}' for k in KEYS)
print(f'\n{"="*(20 + col_w * len(KEYS))}')
print(header)
print(f'  {"-"*(18 + col_w * len(KEYS))}')

# Generates the summary table using the nested metrics dictionary.
for frac, modes in sweep.items():
    label = ('unsupervised'   if frac == 0.0 else
             'fully supervised' if frac == 1.0 else
             f'{frac:.0%} labels')
    row = f'  {label:>18}'
    for k in KEYS:
        acc = f'{modes[k]["metrics"]["test_acc"]*100:.2f}%' if k in modes else 'n/a'
        row += f'{acc:>{col_w}}'
    print(row)
print(f'{"="*(20 + col_w * len(KEYS))}\n')


# ── Individual Accuracy per Epoch Plots ───────────────────────────────

def plot_accuracy_comparison(fraction: float, model_type: str, baseline_history: dict, vat_history: dict):
    """
    Creates a single line chart comparing the accuracy of a Baseline model and a VAT model.
    The chart places epochs on the X-axis and accuracy percentage on the Y-axis.
    Solid lines represent validation accuracy. Dashed lines represent training accuracy.
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    title = f'{model_type} Accuracy Comparison ({fraction * 100}% Labeled Data)'
    ax.set_title(title, fontweight='bold')

    # Plots Baseline data if the history is provided.
    if baseline_history:
        epochs_base = range(1, len(baseline_history['train_acc']) + 1)
        ax.plot(epochs_base, [v * 100 for v in baseline_history['train_acc']],
                '--', color='steelblue', alpha=0.5, label='Baseline Train')
        ax.plot(epochs_base, [v * 100 for v in baseline_history['val_acc']],
                '-', color='steelblue', linewidth=2, label='Baseline Val')

    # Plots VAT data if the history is provided.
    if vat_history:
        epochs_vat = range(1, len(vat_history['train_acc']) + 1)
        ax.plot(epochs_vat, [v * 100 for v in vat_history['train_acc']],
                '--', color='darkorange', alpha=0.5, label='VAT Train')
        ax.plot(epochs_vat, [v * 100 for v in vat_history['val_acc']],
                '-', color='darkorange', linewidth=2, label='VAT Val')

    ax.set_xlabel('Epochs')
    ax.set_ylabel('Accuracy (%)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

def generate_all_plots(fractions: list, model_types: list, sweep_data: dict):
    """
    Iterates through all provided data fractions and model types to generate individual plots.
    Retrieves the training history from the nested sweep_data dictionary.
    """
    for frac in fractions:
        for model in model_types:
            key_baseline = f'{model} / baseline'
            key_vat = f'{model} / vat'

            base_data = sweep_data.get(frac, {}).get(key_baseline, {})
            vat_data = sweep_data.get(frac, {}).get(key_vat, {})

            hist_base = base_data.get('history')
            hist_vat = vat_data.get('history')

            if hist_base or hist_vat:
                plot_accuracy_comparison(frac, model, hist_base, hist_vat)

# Defines the models and fractions based on the initial configuration.
model_types = ['FCN', 'SmallCNN']
FRACTIONS = [0.0, 0.25, 0.5, 1.0]

# Executes the plotting sequence.
generate_all_plots(FRACTIONS, model_types, sweep)